# STEP 1 — QC UTCI hourly Jawa (1996–2025)
**Audit lengkap file sumber.** Adaptasi dari `UTCI_JAWA_STEP1_QC_1996_2025.ipynb` dan QC Jawa. Belum melakukan crop atau konversi. Memeriksa 24 jam UTC, konsistensi grid, tanggal, kesesuaian unit dan duplikasi.

**Folder input:** `D:\Daps\Data\UTCI\1996-2025`  
**Folder output semua tahap:** `D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025`

Jalankan sel dari atas ke bawah; status wajib **LULUS** sebelum STEP 2. File CSV menyimpan seluruh hasil audit.

In [1]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

ROOT = Path(r"D:\Daps\Data\UTCI")
INPUT_DIR = ROOT / "1996-2025"
OUT_DIR = ROOT / "Hasil_UTCI_Jawa_1996_2025"
START_YEAR, END_YEAR = 1996, 2025
QC_RAW = OUT_DIR / "01_qc_utci_raw_files_1996_2025.csv"
QC_YEAR = OUT_DIR / "02_qc_utci_calendar_per_year_1996_2025.csv"
QC_STATUS = OUT_DIR / "02_qc_utci_step1_status.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Folder input tidak ditemukan: {INPUT_DIR}")
print('Input :', INPUT_DIR, '\nOutput:', OUT_DIR)

Input : D:\Daps\Data\UTCI\1996-2025 
Output: D:\Daps\Data\UTCI\Hasil_UTCI_Jawa_1996_2025


## 1. Daftar file sumber — input hanya dari folder raw, bukan folder hasil

In [2]:
files = sorted(p for p in INPUT_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.nc')
expected_days = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31', freq='D')
if not files: raise FileNotFoundError(f'Tidak ada file .nc di {INPUT_DIR}')
print(f'NetCDF ditemukan: {len(files):,}; jumlah hari ideal: {len(expected_days):,}')
print('Contoh:', *[p.name for p in files[:3]], sep='\n • ')

NetCDF ditemukan: 10,958; jumlah hari ideal: 10,958
Contoh:
 • ECMWF_utci_19960101_v1.1_con.area-subset.-5.5.115.-9.104.nc
 • ECMWF_utci_19960102_v1.1_con.area-subset.-5.5.115.-9.104.nc
 • ECMWF_utci_19960103_v1.1_con.area-subset.-5.5.115.-9.104.nc


## 2. Pemeriksaan file per file — satu file dibuka dan ditutup langsung

In [3]:
def get_da(ds):
    var = next((k for k in ['utci', 'UTCI'] if k in ds.data_vars), None)
    if var is None: raise ValueError(f'Variabel utci tidak ditemukan, tersedia {list(ds.data_vars)}')
    da = ds[var]
    rename = {}
    for opts, target in [(('latitude', 'lat'), 'latitude'), (('longitude', 'lon'), 'longitude'), (('time', 'valid_time'), 'time')]:
        key = next((x for x in opts if x in da.dims), None)
        if key is None: raise ValueError(f'Dimensi {target} hilang dari {da.dims}')
        if key != target: rename[key] = target
    return da.rename(rename).transpose('time', 'latitude', 'longitude')


def inspect_file(path, reference=None):
    with xr.open_dataset(path, decode_times=True, decode_timedelta=False) as ds:
        da = get_da(ds)
        lat = np.asarray(da.latitude.values, dtype=float)
        lon = np.asarray(da.longitude.values, dtype=float)
        times = pd.DatetimeIndex(da.time.values)
        grid_ok = reference is None or (lat.shape == reference[0].shape and
                   lon.shape == reference[1].shape and np.allclose(lat, reference[0], atol=1e-6) and
                   np.allclose(lon, reference[1], atol=1e-6))
        one_date = (len(times) > 0 and len(times.normalize().unique()) == 1)
        hourly_ok = (len(times) == 24 and np.array_equal(
            np.diff(times.values), np.full(23, np.timedelta64(1, 'h'))))
        # load only per-file reductions; memory is released at the end of with
        mi, ma, me = (float(x.values) for x in (da.min(skipna=True), da.max(skipna=True), da.mean(skipna=True)))
        units = str(da.attrs.get('units', '')).strip()
        guess = ('Kelvin' if units.lower() in ('k', 'kelvin', 'degrees_kelvin') or 'kelvin' in units.lower()
                 or (not units and me > 100) else 'Celsius_or_unknown')
        row = dict(path=str(path), file=path.name,
                   date=times[0].normalize() if one_date else pd.NaT,
                   time_start=times.min() if len(times) else pd.NaT,
                   time_end=times.max() if len(times) else pd.NaT,
                   n_time=len(times), one_day_file_ok=bool(one_date),
                   hourly_interval_ok=bool(hourly_ok), grid_ok=bool(grid_ok),
                   n_lat=len(lat), n_lon=len(lon), raw_min=mi, raw_max=ma,
                   raw_mean=me, units_attr=units, unit_guess=guess,
                   nan_count=int(da.isnull().sum().values))
    return row, (lat, lon), times.values

rows, all_times, reference = [], [], None
for i, path in enumerate(files, 1):
    row, coords, times = inspect_file(path, reference)
    if reference is None: reference = tuple(x.copy() for x in coords)
    rows.append(row); all_times.append(times)
    if i <= 3 or i % 500 == 0 or i == len(files):
        print(f'[{i:05d}/{len(files):05d}] {path.name} → {row["date"]}')
qc_df = pd.DataFrame(rows).sort_values(['date', 'file'], na_position='last').reset_index(drop=True)
qc_df.to_csv(QC_RAW, index=False)
display(qc_df.head())

[00001/10958] ECMWF_utci_19960101_v1.1_con.area-subset.-5.5.115.-9.104.nc → 1996-01-01 00:00:00
[00002/10958] ECMWF_utci_19960102_v1.1_con.area-subset.-5.5.115.-9.104.nc → 1996-01-02 00:00:00
[00003/10958] ECMWF_utci_19960103_v1.1_con.area-subset.-5.5.115.-9.104.nc → 1996-01-03 00:00:00
[00500/10958] ECMWF_utci_19970514_v1.1_con.area-subset.-5.5.115.-9.104.nc → 1997-05-14 00:00:00
[01000/10958] ECMWF_utci_19980926_v1.1_con.area-subset.-5.5.115.-9.104.nc → 1998-09-26 00:00:00
[01500/10958] ECMWF_utci_20000208_v1.1_con.area-subset.-5.5.115.-9.104.nc → 2000-02-08 00:00:00
[02000/10958] ECMWF_utci_20010622_v1.1_con.area-subset.-5.5.115.-9.104.nc → 2001-06-22 00:00:00
[02500/10958] ECMWF_utci_20021104_v1.1_con.area-subset.-5.5.115.-9.104.nc → 2002-11-04 00:00:00
[03000/10958] ECMWF_utci_20040318_v1.1_con.area-subset.-5.5.115.-9.104.nc → 2004-03-18 00:00:00
[03500/10958] ECMWF_utci_20050731_v1.1_con.area-subset.-5.5.115.-9.104.nc → 2005-07-31 00:00:00
[04000/10958] ECMWF_utci_20061213_v1.1_c

,path,file,date,time_start,time_end,n_time,one_day_file_ok,hourly_interval_ok,grid_ok,n_lat,n_lon,raw_min,raw_max,raw_mean,units_attr,unit_guess,nan_count
0,D:\Daps\Data\UTCI\1996-2025\ECMWF_utci_1996010...,ECMWF_utci_19960101_v1.1_con.area-subset.-5.5....,1996-01-01,1996-01-01,1996-01-01 23:00:00,24,True,True,True,15,45,290.093262,311.308289,299.337891,,Kelvin,0
1,D:\Daps\Data\UTCI\1996-2025\ECMWF_utci_1996010...,ECMWF_utci_19960102_v1.1_con.area-subset.-5.5....,1996-01-02,1996-01-02,1996-01-02 23:00:00,24,True,True,True,15,45,288.946472,310.453003,298.853058,,Kelvin,0
2,D:\Daps\Data\UTCI\1996-2025\ECMWF_utci_1996010...,ECMWF_utci_19960103_v1.1_con.area-subset.-5.5....,1996-01-03,1996-01-03,1996-01-03 23:00:00,24,True,True,True,15,45,288.959015,309.234375,298.467621,,Kelvin,0
3,D:\Daps\Data\UTCI\1996-2025\ECMWF_utci_1996010...,ECMWF_utci_19960104_v1.1_con.area-subset.-5.5....,1996-01-04,1996-01-04,1996-01-04 23:00:00,24,True,True,True,15,45,288.962158,309.608795,298.610291,,Kelvin,0
4,D:\Daps\Data\UTCI\1996-2025\ECMWF_utci_1996010...,ECMWF_utci_19960105_v1.1_con.area-subset.-5.5....,1996-01-05,1996-01-05,1996-01-05 23:00:00,24,True,True,True,15,45,288.743225,311.166565,299.437531,,Kelvin,0


## 3. QC kalender dan timestamp UTC — tanpa duplikasi

In [4]:
file_dates = pd.DatetimeIndex(pd.to_datetime(qc_df.date, errors='coerce').dropna()).normalize()
missing_days = expected_days.difference(file_dates)
extra_days = file_dates.difference(expected_days)
dup_days = file_dates[file_dates.duplicated()]
observed = pd.DatetimeIndex(np.concatenate(all_times))
expected_hours = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31 23:00', freq='h')
dup_hours = observed[observed.duplicated()]
missing_hours = expected_hours.difference(observed)
extra_hours = observed.difference(expected_hours)
year_df = pd.DataFrame([dict(year=y, expected_days=366 if calendar.isleap(y) else 365,
                             observed_days=int((file_dates.year==y).sum())) for y in range(START_YEAR,END_YEAR+1)])
year_df['OK'] = year_df.expected_days.eq(year_df.observed_days)
year_df.to_csv(QC_YEAR, index=False)
print('Tanggal hilang/duplikat/ekstra:', len(missing_days), len(dup_days), len(extra_days))
print('Hourly hilang/duplikat/ekstra :', len(missing_hours), len(dup_hours), len(extra_hours))
print('Tanggal pertama/terakhir     :', file_dates.min(), file_dates.max())
display(year_df)

Tanggal hilang/duplikat/ekstra: 0 0 0
Hourly hilang/duplikat/ekstra : 0 0 0
Tanggal pertama/terakhir     : 1996-01-01 00:00:00 2025-12-31 00:00:00


,year,expected_days,observed_days,OK
0,1996,366,366,True
1,1997,365,365,True
2,1998,365,365,True
3,1999,365,365,True
4,2000,366,366,True
5,2001,365,365,True
6,2002,365,365,True
7,2003,365,365,True
8,2004,366,366,True
9,2005,365,365,True


## 4. Status akhir — data tidak diproses ketika QC gagal

In [5]:
checks = {
    'jumlah_file': len(qc_df) == len(expected_days),
    '24_timestep_per_file': bool(qc_df.n_time.eq(24).all()),
    'interval_1_jam': bool(qc_df.hourly_interval_ok.all()),
    'satu_hari_per_file': bool(qc_df.one_day_file_ok.all()),
    'grid_konsisten': bool(qc_df.grid_ok.all()),
    'tanggal_duplikat_0': len(dup_days) == 0,
    'tanggal_hilang_0': len(missing_days) == 0,
    'tanggal_ekstra_0': len(extra_days) == 0,
    'hourly_duplikat_0': len(dup_hours) == 0,
    'hourly_missing_0': len(missing_hours) == 0,
    'hourly_extra_0': len(extra_hours) == 0,
    'tidak_ada_tanggal_invalid': bool(qc_df.date.notna().all()),
}
status = pd.DataFrame({'check':list(checks), 'passed':list(checks.values())})
status.to_csv(QC_STATUS, index=False)
display(status)
if not all(checks.values()):
    raise RuntimeError(f'QC STEP 1 GAGAL; tinjau {QC_RAW}, {QC_YEAR}, {QC_STATUS}')
print('✅ STEP 1 LULUS — file raw, grid, dan semua timestamp lengkap.')

,check,passed
0,jumlah_file,True
1,24_timestep_per_file,True
2,interval_1_jam,True
3,satu_hari_per_file,True
4,grid_konsisten,True
5,tanggal_duplikat_0,True
6,tanggal_hilang_0,True
7,tanggal_ekstra_0,True
8,hourly_duplikat_0,True
9,hourly_missing_0,True


✅ STEP 1 LULUS — file raw, grid, dan semua timestamp lengkap.
